# Foto → Producto · Notebook 2: upload the fine-tuned model to Hugging Face

Mirrors the course notebook *05b_Upload_SavedModel*. Run it **after** notebook 1.

| Step | What happens |
|---|---|
| 1 | Bring the saved model (`fine_tuned_model.zip`) from Google Drive into Colab |
| 2 | Create the model and image-processor objects from the saved folder |
| 3 | Log in to Hugging Face and push the model, the processor and the prepared dataset |
| 4 | Check: download the model back from Hugging Face and classify a few photos |

You need a Hugging Face **write** token (huggingface.co → Settings → Access Tokens).
This notebook does not need a GPU.

# Step 0: Installation of packages

In [1]:
!pip install "transformers<5.0.0" datasets -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 48.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


# Step 1: Upload the saved model to Colab

Notebook 1 copied `fine_tuned_model.zip` (and `dataset_files.zip`) to `MyDrive/Models/`.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!unzip -o -q /content/drive/MyDrive/Models/fine_tuned_model.zip -d fine_tuned_model
!unzip -o -q /content/drive/MyDrive/Models/dataset_files.zip -d data
!ls fine_tuned_model data

data:
images	test.csv  train.csv  validation.csv

fine_tuned_model:
config.json  model.safetensors	preprocessor_config.json  training_args.bin


# Step 2: Create the model and image-processor objects

In [4]:
from transformers import AutoModelForImageClassification, AutoImageProcessor

model = AutoModelForImageClassification.from_pretrained("fine_tuned_model")
processor = AutoImageProcessor.from_pretrained("fine_tuned_model")

print("Categories:", list(model.config.id2label.values()))

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Categories: ['Blusa', 'Bolsa', 'Calcetines', 'Camisa', 'Jeans', 'Lentes de sol', 'Mochila', 'Pantalón', 'Playera', 'Reloj', 'Sandalias', 'Short', 'Tenis', 'Vestido']


# Step 3: Upload to Hugging Face

Paste your **write** token when asked (answer `n` to "Add token as git credential?").

In [6]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `hf auth whoami` to get more information or `hf auth logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: write).
The token `ke

In [7]:
!hf auth whoami

user:  boofalcon


In [8]:
MODEL_REPO = "boofalcon/foto-a-producto-vit"          # <Hugging Face account>/<model name>
DATASET_REPO = "boofalcon/foto-a-producto-dataset"

model.push_to_hub(MODEL_REPO)
processor.push_to_hub(MODEL_REPO)          # the app needs the processor together with the model
print("Model:", "https://huggingface.co/" + MODEL_REPO)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...6zygc2y/model.safetensors:   0%|          | 1.11MB /  343MB            

README.md: 0.00B [00:00, ?B/s]

Model: https://huggingface.co/boofalcon/foto-a-producto-vit


Also publish the prepared dataset (the exact train / validation / test split used for fine-tuning),
so anyone can check its size and class balance in the Hugging Face dataset viewer.

In [9]:
import pandas as pd
from datasets import load_dataset, DatasetDict, Image

def load_split(split):
    """The exported CSV of one split, with its image files attached."""
    df = pd.read_csv(f"data/{split}.csv")
    df["image"] = "data/" + df["file"]
    ds = load_dataset("csv", data_files={split: f"data/{split}.csv"})[split]
    ds = ds.add_column("image", df["image"].tolist()).cast_column("image", Image())
    return ds.remove_columns(["file"])

prepared = DatasetDict({s: load_split(s) for s in ["train", "validation", "test"]})
print(prepared)
prepared.push_to_hub(DATASET_REPO)
print("Dataset:", "https://huggingface.co/datasets/" + DATASET_REPO)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'category', 'label', 'articleType', 'baseColour', 'productDisplayName', 'image'],
        num_rows: 4548
    })
    validation: Dataset({
        features: ['id', 'category', 'label', 'articleType', 'baseColour', 'productDisplayName', 'image'],
        num_rows: 930
    })
    test: Dataset({
        features: ['id', 'category', 'label', 'articleType', 'baseColour', 'productDisplayName', 'image'],
        num_rows: 962
    })
})


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmp4ny3s6_q.parquet    : 100%|##########| 1.45MB / 1.45MB            

Dataset: https://huggingface.co/datasets/boofalcon/foto-a-producto-dataset


# Step 4: Check the uploaded model

Load the model **from Hugging Face** (not from the local folder) with a pipeline, exactly as the app will.

In [10]:
from transformers import pipeline
from PIL import Image as PILImage

classifier = pipeline("image-classification", model=MODEL_REPO)
test = pd.read_csv("data/test.csv").sample(8, random_state=1)
for _, row in test.iterrows():
    top = classifier(PILImage.open("data/" + row["file"]).convert("RGB"), top_k=1)[0]
    print(f"true: {row['category']:<14} predicted: {top['label']:<14} score: {top['score']:.2f}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

Device set to use cpu


true: Camisa         predicted: Camisa         score: 1.00
true: Tenis          predicted: Tenis          score: 1.00
true: Pantalón       predicted: Pantalón       score: 1.00
true: Vestido        predicted: Vestido        score: 0.59
true: Jeans          predicted: Pantalón       score: 0.98
true: Jeans          predicted: Jeans          score: 0.75
true: Tenis          predicted: Tenis          score: 1.00
true: Camisa         predicted: Camisa         score: 0.99
